In [58]:
import numpy as np
import itertools
import re


def extract_lrs(s):
    lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
    lr = lr_match.group(1) if lr_match else None

    ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
    ti_lr = ti_lr_match.group(1) if ti_lr_match else None

    return lr, ti_lr

def extract_learning_lora_rank(s):
    match = re.search(r'c\.l(\d+)\.', s)
    if match:
        return int(match.group(1))
    else:
        return None



def get_chunk(data, chunk_index, total_chunks=4):
    """Get a specific chunk from the data"""
    chunk_size = len(data) // total_chunks
    remainder = len(data) % total_chunks
    
    # Calculate start position
    start = chunk_index * chunk_size + min(chunk_index, remainder)
    
    # Calculate end position
    extra = 1 if chunk_index < remainder else 0
    end = start + chunk_size + extra
    
    return data[start:end]

dataset_name2data_root = {
    'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
    'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
    'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
    'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',


    'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
    'chiquita10': 'data_root/data/real_data/chiquita/chiquita-10',
    'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
    'reese50': 'data_root/data/real_data/reese/reese-50',
    'reese10': 'data_root/data/real_data/reese/reese-10',
    'reeseU3': 'data_root/data/real_data/reese/reese-unseen-3',
    'gout50': 'data_root/data/real_data/gout/gout-50',
    'gout10': 'data_root/data/real_data/gout/gout-10',
    'goutU3': 'data_root/data/real_data/gout/gout-unseen-3',
    'jooli50': 'data_root/data/real_data/jooli/jooli-50',
    'jooli10': 'data_root/data/real_data/jooli/jooli-10',
    'jooliU3': 'data_root/data/real_data/jooli/jooli-unseen-3',
    'honer50': 'data_root/data/real_data/honer/honer-50',
    'honer10': 'data_root/data/real_data/honer/honer-10',
    'honerU3': 'data_root/data/real_data/honer/honer-unseen-3',
    'avp20': 'data_root/data/real_data/avp/avp-20',
    'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    

}

dataset_name2data_root['sceleb5g0N50'] = ','.join([dataset_name2data_root[f'{d}50'] for d in ['chiquita','reese','jooli','gout','honer'] ])
dataset_name2data_root['sceleb5g0N10'] = ','.join([dataset_name2data_root[f'{d}10'] for d in ['chiquita','reese','jooli','gout','honer'] ])
dataset_name2data_root['sceleb5g0U3'] = ','.join([dataset_name2data_root[f'{d}U3'] for d in ['chiquita','reese','jooli','gout','honer'] ])

concept2prompt = {
    'crybaby': 'A photo of a crybaby art toy',
    'moodeng': 'A photo of a cute baby hippo',
}
concept2generalprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    
}
concept2initializer = {
    'crybaby': 'toy',
    'moodeng': 'hippo',
    'chiquita': 'person', 
    'reese': 'person', 
    'jooli': 'person', 
    'honer': 'person', 
    'gout': 'person', 
    'avp': 'glasses',
    'sceleb5g0': 'person,person,person,person,person',
}

concept2Prprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    'chiquita': 'A photo of a person',
    'reese': 'A photo of a person',
    'gout': 'A photo of a person',
    'jooli': 'A photo of a person',
    'honer': 'A photo of a person',
    # 'chiquita': 'A photo of a girl',
    'avp': 'A photo of a glasses',
    'sceleb5g0': 'A photo of a person,A photo of a person, A photo of a person, A photo of a person, A photo of a person',
    
}


# delete

concept2domain_preservation_cache_path = {
    'crybaby': 'data_root/cache/mace/general_concept/cache_crybaby.pt',
    'chiquita':'data_root/cache/mace/cache_cele.pt',
    'reese':'data_root/cache/mace/cache_cele.pt',
    'gout':'data_root/cache/mace/cache_cele.pt',
    'jooli':'data_root/cache/mace/cache_cele.pt',
    'honer':'data_root/cache/mace/cache_cele.pt',
    'sceleb5g0':'data_root/cache/mace/cache_cele.pt',
}   

concept2mapping_concept = {
    'crybaby': ['object', 'object'],
    'chiquita': ['person', 'a person'],
    'reese': ['person', 'a person'],
    'gout': ['person', 'a person'],
    'jooli': ['person', 'a person'],
    'honer': ['person', 'a person'],
    
    'sceleb5g0': ['person', """a person','a person','a person','a person','a person"""],
    
}



In [59]:
dataset_name2data_root

{'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
 'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
 'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
 'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
 'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
 'chiquita10': 'data_root/data/real_data/chiquita/chiquita-10',
 'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
 'reese50': 'data_root/data/real_data/reese/reese-50',
 'reese10': 'data_root/data/real_data/reese/reese-10',
 'reeseU3': 'data_root/data/real_data/reese/reese-unseen-3',
 'gout50': 'data_root/data/real_data/gout/gout-50',
 'gout10': 'data_root/data/real_data/gout/gout-10',
 'goutU3': 'data_root/data/real_data/gout/gout-unseen-3',
 'jooli50': 'data_root/data/real_data/jooli/jooli-50',
 'jooli10': 'data_root/data/real_data/jooli/jooli-10',
 'jooliU3': 'data_root/data/real_data/jooli/jooli-unseen-3',
 'honer50': 'data_root/data/real_data/hone

In [4]:

base_exps = [
    
    'ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4'
    #  'c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4',
    #  'c.l64.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4'
     
    # "c.l4.kv_sceleb5g0N50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4"
    # "c.l4.kv_sceleb5g0N50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4"
    
    #     "c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_honer50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_honer50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_jooli50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_jooli50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_jooli50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",


    # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
]
    


exp_names = []
# domain_preservations = ["8e+3"] # ["8e+2","8e+3","8e-4"] # ["8e+2","8e+3","8e-4"]#  ["8e+3"] # ["8e+3"] # ["8e+4","8e+5"] #  ["8e+2","8e+3","8e-4","8e-5"] # 8.0e+3, 8.0e+4 2e-5 
# general_preservations = ["1e-4"]  # ["1e-0","1e-2","1e-4"] # ,"1e-4"] # preservation sclae for the closed-form
# domain_preservations = ["5e-4"] # for 5 concepts
general_preservations = ["1e-4"]
domain_preservations = ["8e+3"]
learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
num_gen_images = [8] # [50] # [8] 
lora_ranks = [1] # [1]
img_types = ["G"] # ["r","g"]
max_train_steps = [50] # [50,200]
# use_prs = [True] # [True, False]

seed_add = 0
manual_target_concept = ""

# sur_concept = 'object'
base_exp_steps = [2000] # we want to see it fit first



for base_exp in base_exps:
    
    
    if not manual_target_concept:
        # Try to infer target_concept from base_exp
        possible_concepts = ['moodeng', 'crybaby', 'avp', 'chiquita', 'reese', 'gout', 'jooli', 'honer', 'sceleb5g0']
        for concept in possible_concepts:
            if concept in base_exp:
                target_concept = concept
                # print(f"target_concept is not set, inferred and set to '{target_concept}' from base_exp")
                break
    else:
        target_concept = manual_target_concept
        # print(f"target_concept is set to '{target_concept}' manually")
    # else:
    #     print("Warning: target_concept is not set and could not be inferred from base_exp.")



    for base_exp_step in base_exp_steps:
        for lr, num_img, lora_rank, img_type, gen_pr, domain_pr, max_train_step in itertools.product(
            learning_rates, num_gen_images, lora_ranks, img_types, general_preservations,domain_preservations, max_train_steps
        ):
            
            
            if 'sd1.4' in base_exp:
                pretrained_model_name_or_path = 'CompVis/stable-diffusion-v1-4'
            elif 'sd1.5' in base_exp:
                pretrained_model_name_or_path = 'runwayml/stable-diffusion-v1-5'
            elif 'ch.' in base_exp: 
                pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
            
            # print(lr, num_img, lora_rank, img_type, steps)
            
            ul_name  = f'ul{lora_rank}.prg{gen_pr}d{domain_pr}.lr{lr}.n{num_img}.{img_type}'
            
            if seed_add > 0:
                exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}.r{seed_add}_{base_exp}.s{base_exp_step}"
                
            else:
                exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}_{base_exp}.s{base_exp_step}"
            final_seed = 2024 + seed_add
            
            mapping_concept = f"['{concept2mapping_concept[target_concept][1]}']"
            config_name =  "erase_default.yaml"
            if 'sceleb5' in target_concept:
                config_name = "erase_sceleb_5.yaml"
            
            
            script = f""" python data_preparation.py configs/custom/{config_name} \\
            exp_name="{exp_name}" \\
            MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
            MACE.num_gen_images={num_img} MACE.seed={final_seed} \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}"
 python training.py configs/custom/{config_name} \\
            exp_name="{exp_name}" \\
            MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
            MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} MACE.seed={final_seed} \\
            MACE.rank={lora_rank} \\
            MACE.num_gen_images={num_img} \\
            MACE.domain_preservation_cache_path={concept2domain_preservation_cache_path[target_concept]} MACE.mapping_concept="{mapping_concept}" \\
            MACE.train_preserve_scale={gen_pr} MACE.preserve_weight={domain_pr} \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}"
            """

            
            print(script)
            # print(exp_name)
            exp_names += [exp_name]
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
            exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000" \
            MACE.pretrained_model_name_or_path="stablediffusionapi/chilloutmix" \
            MACE.num_gen_images=8 MACE.seed=2024 \
            MACE.lora_weight_dir_path="data_root/logs/ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000" \
            MACE.token_embedding_dir_path="data_root/logs/ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000" \
            MACE.input_data_dir="data_root/generated/mace/ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000"
 python training.py configs/custom/erase_default.yaml \
            exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000" \
            MACE.pretrained_model_name_or_path="stablediffusionapi/chilloutmi

In [61]:

base_exps = [
    
    # 'ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-5.ti5e-4_b1g4'
    # 'ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-5.ti5e-4_b1g4'
    
    # 'ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr1e-4.ti5e-4_b1g4'
    # 'ch.c.l16.kv_reese50-V.r_pr1.00.neg_lr1e-4.ti5e-4_b1g4'
    # 'ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr1e-4.ti5e-4_b1g4'
    # 'ch.c.l16.kv_gout50-V.r_pr1.00.neg_lr1e-4.ti5e-4_b1g4'
    
    # 'ch.c.l16.kv_chiquita50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4'
    'ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4'
    # 'ch.c.l16.kv_reese50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4'
    # 'ch.c.l16.kv_gout50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4'

]
    


exp_names = []
# domain_preservations = ["8e+3"] # ["8e+2","8e+3","8e-4"] # ["8e+2","8e+3","8e-4"]#  ["8e+3"] # ["8e+3"] # ["8e+4","8e+5"] #  ["8e+2","8e+3","8e-4","8e-5"] # 8.0e+3, 8.0e+4 2e-5 
# general_preservations = ["1e-4"]  # ["1e-0","1e-2","1e-4"] # ,"1e-4"] # preservation sclae for the closed-form
# domain_preservations = ["5e-4"] # for 5 concepts
general_preservations = ["1e-4"]
domain_preservations = ["8e+3"]
learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
num_gen_images = [8] # [50] # [8] 
lora_ranks = [1] # [1]
img_types = ["G"] # ["r","g"]
max_train_steps = [50] # [50,200]
# use_prs = [True] # [True, False]

# seed_add = 0

seed_adds = [0,1,2]
# seed_adds = [3,4]
manual_target_concept = ""

# sur_concept = 'object'
base_exp_steps = [2000] # we want to see it fit first



for base_exp in base_exps:
    for seed_add in seed_adds:
    
        if not manual_target_concept:
            # Try to infer target_concept from base_exp
            possible_concepts = ['moodeng', 'crybaby', 'avp', 'chiquita', 'reese', 'gout', 'jooli', 'honer', 'sceleb5g0']
            for concept in possible_concepts:
                if concept in base_exp:
                    target_concept = concept
                    # print(f"target_concept is not set, inferred and set to '{target_concept}' from base_exp")
                    break
        else:
            target_concept = manual_target_concept
            # print(f"target_concept is set to '{target_concept}' manually")
        # else:
        #     print("Warning: target_concept is not set and could not be inferred from base_exp.")



        for base_exp_step in base_exp_steps:
            for lr, num_img, lora_rank, img_type, gen_pr, domain_pr, max_train_step in itertools.product(
                learning_rates, num_gen_images, lora_ranks, img_types, general_preservations,domain_preservations, max_train_steps
            ):
                
                
                if 'sd1.4' in base_exp:
                    pretrained_model_name_or_path = 'CompVis/stable-diffusion-v1-4'
                elif 'sd1.5' in base_exp:
                    pretrained_model_name_or_path = 'runwayml/stable-diffusion-v1-5'
                elif 'ch.' in base_exp: 
                    pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
                
                # print(lr, num_img, lora_rank, img_type, steps)
                
                ul_name  = f'ul{lora_rank}.prg{gen_pr}d{domain_pr}.lr{lr}.n{num_img}.{img_type}'
                
                if seed_add > 0:
                    exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}.r{seed_add}_{base_exp}.s{base_exp_step}"
                    
                else:
                    exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}_{base_exp}.s{base_exp_step}"
                final_seed = 2024 + seed_add
                
                mapping_concept = f"['{concept2mapping_concept[target_concept][1]}']"
                config_name =  "erase_default.yaml"
                if 'sceleb5' in target_concept:
                    config_name = "erase_sceleb_5.yaml"
                
                
                script = f""" python data_preparation.py configs/custom/{config_name} \\
                exp_name="{exp_name}" \\
                MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                MACE.num_gen_images={num_img} MACE.seed={final_seed} \\
                MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
                MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
                MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}"
    python training.py configs/custom/{config_name} \\
                exp_name="{exp_name}" \\
                MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} MACE.seed={final_seed} \\
                MACE.rank={lora_rank} \\
                MACE.num_gen_images={num_img} \\
                MACE.domain_preservation_cache_path={concept2domain_preservation_cache_path[target_concept]} MACE.mapping_concept="{mapping_concept}" \\
                MACE.train_preserve_scale={gen_pr} MACE.preserve_weight={domain_pr} \\
                MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}" \\
                MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
                MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}"
                """

                
                print(script)
                # print(exp_name)
                exp_names += [exp_name]
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
                exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000" \
                MACE.pretrained_model_name_or_path="stablediffusionapi/chilloutmix" \
                MACE.num_gen_images=8 MACE.seed=2024 \
                MACE.lora_weight_dir_path="data_root/logs/ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000" \
                MACE.token_embedding_dir_path="data_root/logs/ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000" \
                MACE.input_data_dir="data_root/generated/mace/ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4/checkpoint-2000"
    python training.py configs/custom/erase_default.yaml \
                exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000" \
                MACE.pretrained_model_name_or_path="stablediffusion

In [62]:
# relearning
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50.r1_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50.r2_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000']



use_te = True

#seed = 0

seeds = [0,1,2] # along

# seeds = [3,4]


batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

pretrained = 'ch'
concept = ""
use_pr = True

use_te = True
lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    'data_setting': 'small',
    # 'lora_rank' :  16
}

data_setting = 'full' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name,seed in zip(ul_exp_names,seeds):
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    #fix edit here : they are using this to reconstruct the base_exp as well
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    
    # todo: better use 're'
    for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
        
        # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
        pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

        if not concept:
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            if 'reese' in exp_name: concept = 'reese'
            if 'gout' in exp_name: concept = 'gout'
            if 'jooli' in exp_name: concept = 'jooli'
            if 'honer' in exp_name: concept = 'honer'
        
        use_ti = 'ti' in exp_name or '-V' in exp_name 
        # use_pr = 'pr' in exp_name

        if data_setting == 'fewshot':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}U3'
            elif concept == 'avp':
                dataset_name = 'avpS3'
            else:
                dataset_name = f'{concept}U3'
        if data_setting == 'small':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N10'
            else:
                dataset_name = f'{concept}10'
        else:
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N50'
            elif concept == 'avp':
                dataset_name = 'avp20'
            else:
                dataset_name = f'{concept}50'
                
        if reV:
            initializer_token = concept2initializer[concept]
        else: 
            initializer_token = ''
            
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]

        name_tag = ''
        if is_relearn: name_tag += 'uul'
        name_tag = f'{name_tag} {dataset_name}'
        name_tag += f' l{lora_rank}'
        if use_ti: 
            # name_tag += f' ti.{lr_ti}'
            name_tag += f' ti'

        data_root = dataset_name2data_root[dataset_name]
        
        if use_ti:
            dataset_name_for_exp = dataset_name + "-V"
            # if use_ni:
            #     dataset_name_for_exp += ".ni"
        else: dataset_name_for_exp = dataset_name
        
        
        # prior preservation folder
        prior_folder = 'original_realistic_vision'
        if pretrained == 'sd1.5':
            prior_folder = 'original_pretrained_sd1.5'
        if pretrained == 'sd1.4':
            prior_folder = 'original_pretrained_sd1.4'     
        if pretrained == 'rv':
            prior_folder = 'original_realistic_vision'
        elif pretrained == 'ch':
            prior_folder = 'original_chilloutmix'

        
        # renaming to check
        # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
        # if use_pr:
        #     re_exp_name += f'_pr0.50'
        # re_exp_name += '_lr'
        # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
        # if use_ti:
        #     re_exp_name += f'.ti{str(lr_ti)}'
        # re_exp_name += '_f0.5_b1g4'
        
        # print(re_exp_name)
        # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

        # if manual_lora is not None and manual_data != lora_rank:
        
        if use_manual_params:
            lora_rank = re_lora_rank
            eff_data_setting = manual_params['data_setting']

            lr_lora, lr_ti = re_lr_lora, re_lr_ti
        
            if eff_data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            if eff_data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'            
                    
            data_root = dataset_name2data_root[dataset_name]
        
        
        
        
        if reV:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
        else:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
         
           
        if lr_scheduler == 'linear':
            relearn_exp_name += f".ln"
        relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
            
            
        if use_pr:
            relearn_exp_name += f".pr1.00"
            if apply_negative_prompt:
                relearn_exp_name += ".neg"
        
        relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
        
        if seed != 0:
            relearn_exp_name += f".r{seed}"

        final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
        
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
        --pretrained_model_name_or_path={pretrained_path}  \\
        --instance_data_dir={data_root} \\
        --output_dir="data_root/logs/{final_exp_name}" \\
        --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
        --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
        --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
        --run_note '{name_tag}' \\"""
            
        script += f"""
        --cfg_scale 6.0 \\"""
    
    
        if apply_negative_prompt:
            script += f"""
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
            
            
        if use_pr:
            script += f"""
        --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
        --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/6.00" \\"""
            
        # Conditional learning rate + TI options
        if use_ti:
            
            if lora_rank <= 0:
                script += f"""
        --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                if use_te:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
        --learning_rate {lr_lora}"""

        print(script)
        final_exp_names += [final_exp_name]
print(final_exp_names)

        


        accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000/LoRA_fusion_model  \
        --instance_data_dir=data_root/data/real_data/honer/honer-10 \
        --output_dir="data_root/logs/rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000" \
        --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
        --train_batch_size=1 --gradient_accumulation_steps=4 \
        --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
        --run_note 'uul honer50 l16 ti' \
        --cfg_scale 6.0 \
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, 

In [68]:
# exp_names = ['rl4.chiquita50_ul1.prg1e-4d8e-5.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
#    ['uul1.lr1e-4.n8.G.chiquita.obj.s8_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 
    # 'uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
# ul_exp_names = , 'ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']
# ul_exp_names = [, ]
# uul1.prg8e+7d1e-2.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000
exp_names = ['rl16.reV.sceleb5g0N10.lr5e-5.ti5e-4.r2_ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50.r2_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']


exp_names = ['rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000']











exp_names = [exp_names[0]]


# exp_names = 


apply_negative_prompt = True

# exp_names = [exp_names[0]]
count = 0

cfg_scales = [  4.5, 6.0]
cfg_scales = [3.0,4.5,6.0,7.5]

for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name or 'rl' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)
    # steps =range(800, 1000+1, 100)
    # steps =range(300, 500+1, 100)
    for step in steps:
    # for step in [1200,1800]:

        # for cfg in cfg_scales:
        is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
        is_unlearn = 'ul' in exp_name and not is_relearn
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'sceleb5g0' in exp_name: concept = 'sceleb5g0'
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            # erase_name = concept
            # if 'VPr' in exp_name: erase_name += 'VPr'
            # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
        if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            
        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""
    
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        


        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000/LoRA_fusion_model'  \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --load_lora_weight_path="data_root/logs/rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000/checkpoint-0" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d

In [ ]:
exp_names = [
    "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
]



    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [4.5,6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    # for step in range(3100, 4000+1, 100):
    for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
            
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                erase_name = concept
                if 'VPr' in exp_name: erase_name += 'VPr'
                pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name
            
            if 'V.ni' in exp_name:
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]



            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
            
            if 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="data_root/logs/c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4/checkpoint-2000" \
                --gen_image_path="auto" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --load_token_embedding_path="data_root/logs/c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4/checkpoint-2000" \
                --placeholder_token="v1" --initializer_token='girl' \
                --cfg_scale 4.50

            accelerate launch train_dreambooth_lora

In [69]:

# decoding unlearning - with same hyperparameter
ul_exp_names = [
    "ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    
    
    # "ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000"
]
data_setting = 'full' 
use_ni = True
for ul_exp_name in ul_exp_names:
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
    pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

    if 'moodeng' in exp_name: concept = 'moodeng'
    if 'crybaby' in exp_name: concept = 'crybaby'
    if 'avp' in exp_name: concept = 'avp'
    if 'chiquita' in exp_name: concept = 'chiquita'
    use_ti = 'ti' in exp_name or '-V' in exp_name 
    use_pr = 'pr' in exp_name


    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

            
    if use_ni:
        initializer_token = ''
    elif use_ti:
        initializer_token = concept2initializer[concept]
        
    if use_ti:
        prompt = 'A photo of a v1'
    else:
        prompt = concept2prompt[concept]

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'

    data_root = dataset_name2data_root[dataset_name]
    
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        # if use_ni:
        #     dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name
    
    
    # renaming to check
    re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        re_exp_name += f'_pr0.50'
    re_exp_name += '_lr'
    if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
    if use_ti:
        re_exp_name += f'.ti{str(lr_ti)}'
    re_exp_name += '_f0.5_b1g4'
    
    # print(re_exp_name)
    assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/u{ul_exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)




        


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=data_root/logs/ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/LoRA_fusion_model  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-50 \
    --output_dir="data_root/logs/uul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
    --run_note ' chiquita50 l4 ti' \
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \
    --class_prompt="A photo of a girl" --class_data_dir="data_root/generated/model/original_pretrained/A photo of a girl/7.50" \
    --learning_rate_lora 5e-4 --learning_rat

In [48]:


# concept = 'sceleb5g0' # moodeng
concept = 'chiquita' # moodeng
data_setting = 'full' # full
is_relearn = False # True

pretrained = 'sd1.5' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 1 #4

lora_rank = 16 # 1
lora_alpha = None  # None


use_pr = True
use_ti = True # True 
use_ni = True

seeds = [0]  # List of seeds

lr_lora_grid = ["5e-4","1e-4","5e-5"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti

# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                seeds))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)

final_exp_names = []
for lr_lora, lr_ti, seed in combos:

    if data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    if data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if lora_alpha:
        exp_name = f'c.l{lora_rank}.kv.a{lora_alpha}_{dataset_name_for_exp}'
    else:
        exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
    exp_name += '_lr'
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    # prior preservation folder
    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 3000
    if data_setting == 'fewshot':
        max_train_steps = 1000
    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
            
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:

        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person/6.00" \\"""    


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-50 \
    --output_dir="data_root/logs/sd15.c.l16.kv_chiquita50-V.r_pr1.00_lr5e-4.ti5e-4_b1g1" \
    --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
    --train_batch_size=1 --gradient_accumulation_steps=1 \
    --lora_rank 16 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=3000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
    --run_note ' chiquita50 l16 ti' \
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_pretrained_sd1.5/a photo of a person/6.00" \
    --cfg_scale 6.0 \
    --learning_rate_lora 5e-4 --learning_rate_ti 5e-4 \
    --placeholder_token="v1" --initializer_token=''

    accelerate launch train_

In [99]:
# hack pretraining

# concept = 'sceleb5g0' # moodeng
concept = 'jooli' # moodeng
data_setting = 'full' # full
is_relearn = False # True

pretrained = 'ch' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 4 #4


lora_rank = 16 # 1
lora_alpha = None  # None

use_te = False
use_pr = True
use_nis = [True]
use_ti = True # True 

seeds = [0]  # List of seeds

lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
# lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
lr_lora_grid = ["5e-4"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti


lr_lora_te_grid = ["1e-5"]
# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                lr_lora_te_grid if use_te else [None],
                                seeds,use_nis ))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)

final_exp_names = []
for lr_lora, lr_ti, lr_te, seed, use_ni in combos:

    if data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    if data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if use_te:
        exp_name = f'ct.l{lora_rank}.kv'
    else: exp_name = f'c.l{lora_rank}.kv'
    
    if lora_alpha:
        exp_name += f'.a{lora_alpha}'
    
    exp_name += f'_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
        
    if lr_scheduler == 'constant':
        exp_name += '_lr'
    elif lr_scheduler == 'linear':
        exp_name += '_ln.lr'
        
        
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 2000
    if data_setting == 'fewshot':
        max_train_steps = 1000

    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
    if lr_scheduler == 'linear' and data_setting == 'small':
        max_train_steps = 1000
    if lr_scheduler == 'linear' and data_setting == 'full':
        max_train_steps = 2000        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --lr_scheduler "{lr_scheduler}" \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:

        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person/6.00" \\"""    


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            if use_te:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""



    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=stablediffusionapi/chilloutmix  \
    --instance_data_dir=data_root/data/real_data/jooli/jooli-50 \
    --output_dir="data_root/logs/ch.c.l16.kv_jooli50-V.r_pr1.00_lr5e-4.ti5e-4_b1g4" \
    --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 16 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=2000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
    --lr_scheduler "constant" \
    --run_note ' jooli50 l16 ti' \
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_chilloutmix/a photo of a person/6.00" \
    --cfg_scale 6.0 \
    --learning_rate_lora 5e-4 --learning_rate_ti 5e-4 \
    --placeholder_token="v1" --initializer_token=''
['ch.c.l16.kv_jo

In [40]:
# hack pretraining

# concept = 'sceleb5g0' # moodeng
concept = 'honer' # moodeng
data_setting = 'full' # full
is_relearn = False # True

pretrained = 'ch' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 4 #4


lora_rank = 16 # 1
lora_alpha = None  # None

use_te = False
use_pr = True
use_nis = [True]
use_ti = True # True 

use_pr_negative = True

seeds = [0]  # List of seeds

lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
# lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
lr_lora_grid = ["5e-5"]
# lr_lora_grid = ["5e-4"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti


lr_lora_te_grid = ["1e-5"]
# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                lr_lora_te_grid if use_te else [None],
                                seeds,use_nis ))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)


apply_negative_prompt = True

final_exp_names = []
for lr_lora, lr_ti, lr_te, seed, use_ni in combos:

    if data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    if data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if use_te:
        exp_name = f'ct.l{lora_rank}.kv'
    else: exp_name = f'c.l{lora_rank}.kv'
    
    if lora_alpha:
        exp_name += f'.a{lora_alpha}'
    
    exp_name += f'_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
        if use_pr_negative:
            exp_name += '.neg'
        
    if lr_scheduler == 'constant':
        exp_name += '_lr'
    elif lr_scheduler == 'linear':
        exp_name += '_ln.lr'
        
        
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 2000
    if data_setting == 'fewshot':
        max_train_steps = 1000

    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
    if lr_scheduler == 'linear' and data_setting == 'small':
        max_train_steps = 1000
    if lr_scheduler == 'linear' and data_setting == 'full':
        max_train_steps = 2000        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --lr_scheduler "{lr_scheduler}" \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        pr_prompt = "a photo of a person"
        if use_pr_negative:
            pr_prompt = "a photo of a person_neg"
        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/{pr_prompt}/6.00" \\"""    


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    
    if apply_negative_prompt:
        script += f"""
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            if use_te:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""



    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=stablediffusionapi/chilloutmix  \
    --instance_data_dir=data_root/data/real_data/honer/honer-50 \
    --output_dir="data_root/logs/ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-5.ti5e-4_b1g4" \
    --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 16 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=2000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
    --lr_scheduler "constant" \
    --run_note ' honer50 l16 ti' \
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_chilloutmix/a photo of a person_neg/6.00" \
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low q

In [70]:
# hack finetuning

# concept = 'sceleb5g0' # moodeng
concept = 'chiquita' # moodeng
data_setting = 'small' # full
is_relearn = False # True

pretrained = 'ch' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 4 #4


lora_rank = 4 # 1
lora_alpha = None  # None

use_te = True
use_pr = True
use_nis = [False]
use_ti = True # True 

use_pr_negative = True

seeds = [0,1,2]  # List of seeds
# seeds = [3,4]  # List of seeds

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'constant'
# lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
lr_lora_grid = ["1e-4"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti


lr_lora_te_grid = ["1e-5"]
# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                lr_lora_te_grid if use_te else [None],
                                seeds,use_nis ))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)


apply_negative_prompt = True

final_exp_names = []
for lr_lora, lr_ti, lr_te, seed, use_ni in combos:

    if data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    if data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if use_te:
        exp_name = f'ct.l{lora_rank}.kv'
    else: exp_name = f'c.l{lora_rank}.kv'
    
    if lora_alpha:
        exp_name += f'.a{lora_alpha}'
    
    exp_name += f'_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
        if use_pr_negative:
            exp_name += '.neg'
        
    if lr_scheduler == 'constant':
        exp_name += '_lr'
    elif lr_scheduler == 'linear':
        exp_name += '_ln.lr'
        
        
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 2000
    if data_setting == 'fewshot':
        max_train_steps = 1000

    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
    if lr_scheduler == 'linear' and data_setting == 'small':
        max_train_steps = 1000
    if lr_scheduler == 'linear' and data_setting == 'full':
        max_train_steps = 2000        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --lr_scheduler "{lr_scheduler}" \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        pr_prompt = "a photo of a person"
        if use_pr_negative:
            pr_prompt = "a photo of a person_neg"
        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/{pr_prompt}/6.00" \\"""    


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    
    if apply_negative_prompt:
        script += f"""
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            if use_te:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""



    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=stablediffusionapi/chilloutmix  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-10 \
    --output_dir="data_root/logs/ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4" \
    --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
    --lr_scheduler "linear" \
    --run_note ' chiquita10 l4 ti' \
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_chilloutmix/a photo of a person_neg/6.00" \
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quali

In [73]:
# # this one
base_exps =['original_realistic_vision'] # 'original_realistic_vision']

# base_exps = ['ch.c.l16.kv_chiquita50-V_pr1.00_lr5e-4.ti5e-4_b1g1']
base_exps = ['original_pretrained_sd1.5']
base_exps = ['ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2']

    # "ch.ct.l4.kv_reese10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4",
    
    # "ch.ct.l4.kv_honer10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1",
    # "ch.ct.l4.kv_honer10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2"
    
base_exps = ['ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2']


base_exps = [base_exps[2]]

# base_exps =['original_chilloutmix'] # 'original_chilloutmix']

apply_negative_prompt = False

counter = 0
# base_exps = ['c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4','c.l64.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4']

 #, 'c.l4.kv_chiquitaU3-V_lr1e-4.ti5e-2_f0.5_b1g4.r4', 'c.l4.kv_chiquitaU3-V_lr5e-5.ti5e-2_f0.5_b1g4.r4']


# base_exps = [base_exps[i] for i in range(0, len(base_exps), 4)] # take every second element
# base_exps = [base_exps[0]]
chunk_id = 0
total_chunks=1
exp_names = get_chunk(base_exps,chunk_id,total_chunks=total_chunks)

    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    # manual_prompt = 'a photo of a person'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()


    
    # steps = range(0, 3001, 200)
    steps = range(0, 1001, 100)
    steps =[500, 1001]
    # steps = range(0, 501, 100)
    # steps = range(1000, 1001, 100)
    # steps = range(2000, 2001, 100)
    # steps = [6000,8000,10000]
    cfg_scales = [3.0,4.5,6.0,7.5]
    cfg_scales = [6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    for step in steps:
    # for step in range(6000, 10001, 100):
    # for step in range(0, 10001, 100):
    # for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        # for cfg in cfg_scales:
        is_original_pretrained_sd14 = exp_name == 'original_pretrained_sd1.4'
        is_original_pretrained_sd15 = exp_name == 'original_pretrained_sd1.5'
        is_original_rv = exp_name == 'original_realistic_vision'
        is_original_ch = exp_name == 'original_chilloutmix'
        is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
        is_unlearn = 'ul' in exp_name and not 'uul' in exp_name
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'reese' in exp_name: concept = 'reese'
        if 'gout' in exp_name: concept = 'gout'
        if 'jooli' in exp_name: concept = 'jooli'
        if 'honer' in exp_name: concept = 'honer'
        if 'sceleb' in exp_name: concept = 'sceleb5g0'
        
        
        if is_original_pretrained_sd14 or 'sd14.' in exp_name:
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_original_pretrained_sd15 or 'sd15.' in exp_name:
            pretrained_path = 'runwayml/stable-diffusion-v1-5'
        if is_original_rv or 'rv.' in exp_name:
            pretrained_path = 'stablediffusionapi/realistic-vision-v51'
        elif is_original_ch or 'ch.' in exp_name:
            pretrained_path = 'stablediffusionapi/chilloutmix'
        
        if is_relearn:
            erase_name = concept
            if 'VPr' in exp_name: erase_name += 'VPr'
            pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = '-V' in exp_name 
        # print(f"use_ti: {use_ti}")
        # print(exp_name)
        
        if 'V.r' in exp_name:
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        
        elif use_ti:
            
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                # prompt = 'A photo of a v1' 
                # prompt = 'v1' 
                
                placeholder_token = 'v1'
                
        
        else:
            if concept in concept2prompt:
                prompt = concept2prompt[concept]
            else: 
                prompt = 'sks person'
        
        if is_unlearn or 'erase' in exp_name or is_original_pretrained_sd14 or is_original_pretrained_sd15 or is_original_rv or is_original_ch: 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            

        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        
        # script += f"""
        #     --cfg_scale {cfg:.2f}"""
        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""
        print(script) 
        
        counter += 1
print(f"Total scripts generated: {counter}")
        
        


        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='stablediffusionapi/chilloutmix'  \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --load_lora_weight_path="data_root/logs/ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2/checkpoint-500" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/ch.ct.l4.kv_chiquita10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2/checkpoint-500" \
            --placeholder_token="v1" --initializer_token='person' \
            --cfg_scale 6.00,7.50

        accelerate launch train_dreambooth_lora.py \
            --pretrained_model

In [ ]:
how to t